In [58]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [59]:
df= pd.read_csv('loan_data.csv')
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [60]:
x=df.drop(columns=['loan_status'])
y=df.loan_status

In [61]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [62]:
obj_cols=x.select_dtypes(include='object').columns
obj_cols

Index(['person_gender', 'person_education', 'person_home_ownership',
       'loan_intent', 'previous_loan_defaults_on_file'],
      dtype='object')

In [63]:
xtrain[obj_cols].nunique()

person_gender                     2
person_education                  5
person_home_ownership             4
loan_intent                       6
previous_loan_defaults_on_file    2
dtype: int64

In [64]:
df['person_education'].unique()

array(['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate'],
      dtype=object)

In [65]:
preprocessing=ColumnTransformer(
    transformers=[
        ('oh_encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
          obj_cols.drop('person_education')),
        ('ord_encoder', OrdinalEncoder(categories=[['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']], handle_unknown='use_encoded_value', unknown_value=-1),['person_education'])
    ],
    remainder='passthrough'
)

In [66]:
main_pipeline=Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('model', DecisionTreeClassifier(random_state=42))
    ]
)

In [67]:
grid_search_cv= GridSearchCV(
    estimator=main_pipeline,
    param_grid={
        'model__criterion' : ['gini', 'entropy'],
        'model__splitter': ['best', 'random'] ,
        'model__max_depth': [None,5,10],
        'model__min_samples_split': [2,5],
        'model__min_samples_leaf': [1,3,5]
    },
    verbose=10,
    n_jobs=-1
)
grid_search_cv.fit(xtrain, ytrain)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [None, 5, ...], 'model__min_samples_leaf': [1, 3, ...], 'model__min_samples_split': [2, 5], ...}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,10
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('oh_encoder', ...), ('ord_encoder', ...)]"


In [68]:
grid_search_cv.best_estimator_

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('oh_encoder', ...), ('ord_encoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [69]:
grid_search_cv.best_params_

{'model__criterion': 'entropy',
 'model__max_depth': 10,
 'model__min_samples_leaf': 1,
 'model__min_samples_split': 2,
 'model__splitter': 'best'}

In [70]:
results=pd.DataFrame(grid_search_cv.cv_results_)

In [71]:
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.552410,0.094983,0.036717,0.002037,gini,None,1,2,best,"{'model__criterion': 'gini', 'model__max_depth...",0.896806,0.900972,0.899444,0.901250,0.895694,0.898833,0.002225,52
1,0.294379,0.044361,0.056246,0.034076,gini,None,1,2,random,"{'model__criterion': 'gini', 'model__max_depth...",0.888333,0.891111,0.884583,0.892639,0.889722,0.889278,0.002749,59
2,0.482562,0.083516,0.037730,0.006944,gini,None,1,5,best,"{'model__criterion': 'gini', 'model__max_depth...",0.895139,0.901806,0.900000,0.901667,0.901250,0.899972,0.002499,49
3,0.278602,0.092414,0.042732,0.013301,gini,None,1,5,random,"{'model__criterion': 'gini', 'model__max_depth...",0.895278,0.897778,0.890000,0.889306,0.895417,0.893556,0.003315,57
4,0.457931,0.050133,0.034622,0.006849,gini,None,3,2,best,"{'model__criterion': 'gini', 'model__max_depth...",0.898194,0.903333,0.906944,0.901111,0.902639,0.902444,0.002861,43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,0.162005,0.006139,0.031619,0.002077,entropy,10,3,5,random,"{'model__criterion': 'entropy', 'model__max_de...",0.903472,0.908889,0.905417,0.911389,0.906944,0.907222,0.002740,30
68,0.303512,0.020627,0.033123,0.004065,entropy,10,5,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918472,0.927361,0.914444,0.921250,0.920417,0.920389,0.004204,1
69,0.171082,0.020035,0.032827,0.000814,entropy,10,5,2,random,"{'model__criterion': 'entropy', 'model__max_de...",0.910694,0.899444,0.899583,0.911528,0.912083,0.906667,0.005857,32
70,0.282431,0.016805,0.020438,0.005698,entropy,10,5,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918472,0.927361,0.914444,0.921250,0.920417,0.920389,0.004204,1


In [72]:
results.sort_values(by='rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
68,0.303512,0.020627,0.033123,0.004065,entropy,10,5,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918472,0.927361,0.914444,0.921250,0.920417,0.920389,0.004204,1
60,0.302424,0.023187,0.033129,0.001167,entropy,10,1,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.919583,0.926111,0.914861,0.920833,0.920556,0.920389,0.003583,1
70,0.282431,0.016805,0.020438,0.005698,entropy,10,5,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918472,0.927361,0.914444,0.921250,0.920417,0.920389,0.004204,1
62,0.290683,0.006707,0.033105,0.004562,entropy,10,1,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.919583,0.925833,0.914861,0.920556,0.920694,0.920306,0.003491,4
66,0.300590,0.005640,0.031568,0.004107,entropy,10,3,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.919167,0.926389,0.915417,0.920139,0.920000,0.920222,0.003530,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19,0.234435,0.077787,0.067405,0.068871,gini,5,3,5,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872361,0.871667,0.872500,0.872639,0.872444,0.000453,67
13,0.257424,0.051163,0.074739,0.045536,gini,5,1,2,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872361,0.871667,0.872500,0.872639,0.872444,0.000453,67
15,0.304733,0.116977,0.054103,0.035473,gini,5,1,5,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872361,0.871667,0.872500,0.872639,0.872444,0.000453,67
21,0.287183,0.100329,0.039042,0.008628,gini,5,5,2,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872361,0.871667,0.872500,0.872639,0.872444,0.000453,67


In [73]:
# try writing the grid search cv for logistic regression and RandomForest